# B2 · Regression random forest on house prices

**Type**: Implement with scikit-learn · **Level**: ●●○ independent · **Time**: about 55 minutes · **core path ★**

## Objective

Compare on California the linear regression, the geographic KNN (M5), an optimized tree (M7) and the random forest, looking at how the error changes with `n_estimators`, `max_features` and `min_samples_leaf`. Then measure the feature importance by permutation and the cost in time and memory.

## Dataset

`ds.california()` (20,640 districts), 75/25 split with `random_state=0`; RMSE with `squared=False`; the internal estimate is the OOB (`RandomForestRegressor(oob_score=True)` gives the R²).

## Educational purpose

See how much a forest gains over a tree (and over simpler models) on a real problem with strong interactions, and understand which parameters really matter: `max_features` and `min_samples_leaf`, not `n_estimators` beyond a certain point.

## Task

1. Save `rmse` (dictionary) on the test set of: `linear`, `geo knn` (`KNeighborsRegressor(5)` on `Latitude`, `Longitude`), `tree` (`DecisionTreeRegressor(min_samples_leaf=20)`) and `forest` (`RandomForestRegressor(200, max_features=0.33, min_samples_leaf=3, random_state=0, n_jobs=-1)`).
2. With `RandomForestRegressor(100, oob_score=True, ...)` compute `oob_r2` for `max_features` in `[0.2, 0.33, 0.5, 1.0]` and `min_samples_leaf` in `[1, 3, 10]` (`oob_table`, DataFrame with `max_features` rows, `min_samples_leaf` columns); save `best_mf` and `best_leaf` (the argmax).
3. Train the forest with the best values and save `forest_rmse` on the test set, `t_fit` and `t_predict` (seconds for `fit` and for `predict` on the test set) and `size_mb` (size in MB of `pickle.dumps(model)`).
4. Compute `perm` (`pd.Series`, permutation importance on the test set with `n_repeats=5`, `random_state=0`, scoring `neg_root_mean_squared_error`; sorted in descending order) and `top3` (the names of the first three features).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import pickle
import time
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor

X, y = ds.california()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0)
error = lambda m, cols=None: mean_squared_error(y_te, m.fit(X_tr if cols is None else X_tr[cols], y_tr).predict(X_te if cols is None else X_te[cols]), squared=False)

## Your work

In [ ]:
rmse = {}
rmse

In [ ]:
oob_table = pd.DataFrame(index=[0.2, 0.33, 0.5, 1.0], columns=[1, 3, 10], dtype=float)
best_mf = None
best_leaf = None

In [ ]:
forest_rmse = None
t_fit = None
t_predict = None
size_mb = None

In [ ]:
perm = None
top3 = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='Permutation: `r = permutation_importance(model, X_te, y_te, n_repeats=5, random_state=0, scoring="neg_root_mean_squared_error")`; `pd.Series(r.importances_mean, index=X.columns).sort_values(ascending=False)`. Size: `len(pickle.dumps(model)) / 1e6`. Times: `time.time()` before and after `fit` and `predict`.'):
    check(set(rmse) == {"linear", "geo knn", "tree", "forest"} and rmse["forest"] < rmse["tree"] - 0.1 and rmse["forest"] < rmse["geo knn"], "the forest beats the tree by over 0.1 and the geographic KNN", "compare the RMSEs")
    check(rmse["forest"] < 0.55 and rmse["forest"] < rmse["linear"] - 0.15, "forest RMSE below 0.55 and over 0.15 better than the linear one", "RandomForestRegressor(200, max_features=0.33, min_samples_leaf=3)")
    check(oob_table.notna().all().all() and best_mf in (0.33, 0.5) and oob_table.loc[1.0].max() < oob_table.loc[0.33].max(), "the best is 0.33 or 0.5; with all the features (bagging) the OOB R2 is worse", "oob_score_ (R2) of RandomForestRegressor")
    check(best_leaf in (1, 3), "with 100 trees the best leaf is small (1 or 3)", "argmax of the table")
    check(forest_rmse < 0.52 and t_fit > t_predict and 20 < size_mb < 2000, "final RMSE below 0.52; training slower than prediction; model of tens or hundreds of MB", "pickle.dumps(model)")
    check(top3[0] == "MedInc" and set(top3) >= {"MedInc"} and len(perm) == 8 and (perm.diff().dropna() <= 1e-12).all(), "the median income is the most important feature; sorted series", "permutation_importance with scoring neg_root_mean_squared_error")
done()

---
The commented solution is in `solutions/B_with_sklearn/B2_regression_forest_on_california.ipynb`. Open it after you pass the check, or after you have really tried.